# Persian medical QA — open medical LLMs

Asks the same 2 Persian medical questions to every size of:
Apollo2, Apollo-MoE, MedGemma, BioMistral-7B, BiMediX / BiMediX2, Meditron, OpenBioLLM.

Nothing is downloaded: models are called through the Hugging Face router's
OpenAI-compatible API (`https://router.huggingface.co/v1`) as `<repo>:<PROVIDER>`.
Needs `HF_TOKEN` in the environment with `PROVIDER` enabled at
huggingface.co/settings/inference-providers. Models the provider doesn't serve come back
as an error row.

## HF token

In [16]:
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

HF_TOKEN:  ········


## Config

In [17]:
QUESTIONS = [
    (
        "من ۳۵ ساله هستم و سه روز است که تب، سرفه خشک و درد عضلانی دارم. "
        "چه بیماری‌هایی ممکن است عامل آن باشند و چه زمانی باید به پزشک مراجعه کنم؟"
    ),
    (
        "عوارض جانبی شایع داروی متفورمین چیست و "
        "آیا مصرف آن برای بیماران مبتلا به نارسایی کلیه مناسب است؟"
    ),
]

# (family, repo_id, size in B params)
MODELS = [
    # Apollo2
    ("Apollo2", "FreedomIntelligence/Apollo2-0.5B", 0.5),
    ("Apollo2", "FreedomIntelligence/Apollo2-1.5B", 1.5),
    ("Apollo2", "FreedomIntelligence/Apollo2-2B", 2),
    ("Apollo2", "FreedomIntelligence/Apollo2-3.8B", 3.8),
    ("Apollo2", "FreedomIntelligence/Apollo2-7B", 7),
    ("Apollo2", "FreedomIntelligence/Apollo2-9B", 9),
    # Apollo-MoE
    ("Apollo-MoE", "FreedomIntelligence/Apollo-MoE-0.5B", 0.5),
    ("Apollo-MoE", "FreedomIntelligence/Apollo-MoE-1.5B", 1.5),
    ("Apollo-MoE", "FreedomIntelligence/Apollo-MoE-7B", 7),
    # MedGemma
    ("MedGemma", "google/medgemma-4b-it", 4),
    ("MedGemma", "google/medgemma-4b-pt", 4),
    ("MedGemma", "google/medgemma-27b-text-it", 27),
    ("MedGemma", "google/medgemma-27b-it", 27),
    # BioMistral
    ("BioMistral", "BioMistral/BioMistral-7B", 7),
    ("BioMistral", "BioMistral/BioMistral-7B-DARE", 7),
    ("BioMistral", "BioMistral/BioMistral-7B-TIES", 7),
    ("BioMistral", "BioMistral/BioMistral-7B-SLERP", 7),
    # BiMediX / BiMediX2
    ("BiMediX", "BiMediX/BiMediX-Bi", 47),
    ("BiMediX", "BiMediX/BiMediX-Eng", 47),
    ("BiMediX", "BiMediX/BiMediX-Ara", 47),
    ("BiMediX2", "MBZUAI/BiMediX2-8B-hf", 8),
    # Meditron
    ("Meditron", "epfl-llm/meditron-7b", 7),
    ("Meditron", "epfl-llm/meditron-70b", 70),
    ("Meditron3", "OpenMeditron/Meditron3-Gemma2-2B", 2),
    ("Meditron3", "OpenMeditron/Meditron3-Qwen2.5-7B", 7),
    ("Meditron3", "OpenMeditron/Meditron3-8B", 8),
    ("Meditron3", "OpenMeditron/Meditron3-Gemma2-9B", 9),
    ("Meditron3", "OpenMeditron/Meditron3-70B", 70),
    # OpenBioLLM
    ("OpenBioLLM", "aaditya/Llama3-OpenBioLLM-8B", 8),
    ("OpenBioLLM", "aaditya/Llama3-OpenBioLLM-70B", 70),
]

MAX_NEW_TOKENS = 512
PROVIDER = "featherless-ai"

## Runner

In [18]:
import os
import time

from openai import OpenAI

client = OpenAI(base_url="https://router.huggingface.co/v1", api_key=os.environ["HF_TOKEN"])


def ask(repo, question):
    completion = client.chat.completions.create(
        model=f"{repo}:{PROVIDER}",
        messages=[{"role": "user", "content": question}],
        max_tokens=MAX_NEW_TOKENS,
        temperature=0,
    )
    return completion.choices[0].message.content.strip()

In [19]:
results = []
for family, repo, size_b in MODELS:
    print(f"=== {repo} ===")
    for qi, q in enumerate(QUESTIONS, 1):
        row = {"family": family, "model": repo, "size_b": size_b, "q": qi}
        t = time.time()
        try:
            row |= {"answer": ask(repo, q), "error": None}
        except Exception as e:
            row |= {"answer": None, "error": repr(e)}
        row["secs"] = round(time.time() - t, 1)
        results.append(row)
        print(f"--- Q{qi} ({row['secs']}s)\n{row['answer'] or row['error']}\n")

=== FreedomIntelligence/Apollo2-0.5B ===
--- Q1 (1.2s)
BadRequestError('Error code: 400 - {\'error\': {\'message\': "The requested model \'FreedomIntelligence/Apollo2-0.5B\' is not a chat model.", \'type\': \'invalid_request_error\', \'param\': \'model\', \'code\': \'model_not_supported\'}}')

--- Q2 (0.3s)
BadRequestError('Error code: 400 - {\'error\': {\'message\': "The requested model \'FreedomIntelligence/Apollo2-0.5B\' is not a chat model.", \'type\': \'invalid_request_error\', \'param\': \'model\', \'code\': \'model_not_supported\'}}')

=== FreedomIntelligence/Apollo2-1.5B ===
--- Q1 (0.3s)
BadRequestError('Error code: 400 - {\'error\': {\'message\': "The requested model \'FreedomIntelligence/Apollo2-1.5B\' is not a chat model.", \'type\': \'invalid_request_error\', \'param\': \'model\', \'code\': \'model_not_supported\'}}')

--- Q2 (0.3s)
BadRequestError('Error code: 400 - {\'error\': {\'message\': "The requested model \'FreedomIntelligence/Apollo2-1.5B\' is not a chat model.", 

## Results

In [20]:
import pandas as pd
from IPython.display import HTML, display

df = pd.DataFrame(results)
df.to_csv("persian_medical_qa_results.csv", index=False)
display(df[df.error.notna()][["model", "q", "error"]])

ok = df[df.error.isna()].pivot(index=["family", "model", "size_b"], columns="q", values="answer")
ok.columns = [f"Q{c}" for c in ok.columns]
display(HTML(f'<div dir="rtl" style="text-align:right">{ok.to_html()}</div>'))

,model,q,error
0,FreedomIntelligence/Apollo2-0.5B,1,BadRequestError('Error code: 400 - {\'error\':...
1,FreedomIntelligence/Apollo2-0.5B,2,BadRequestError('Error code: 400 - {\'error\':...
2,FreedomIntelligence/Apollo2-1.5B,1,BadRequestError('Error code: 400 - {\'error\':...
3,FreedomIntelligence/Apollo2-1.5B,2,BadRequestError('Error code: 400 - {\'error\':...
4,FreedomIntelligence/Apollo2-2B,1,BadRequestError('Error code: 400 - {\'error\':...
5,FreedomIntelligence/Apollo2-2B,2,BadRequestError('Error code: 400 - {\'error\':...
6,FreedomIntelligence/Apollo2-3.8B,1,BadRequestError('Error code: 400 - {\'error\':...
7,FreedomIntelligence/Apollo2-3.8B,2,BadRequestError('Error code: 400 - {\'error\':...
8,FreedomIntelligence/Apollo2-7B,1,BadRequestError('Error code: 400 - {\'error\':...
9,FreedomIntelligence/Apollo2-7B,2,BadRequestError('Error code: 400 - {\'error\':...
